In [0]:
from pyspark.sql.functions import current_timestamp


source_path = "/Volumes/deltalakehouse/bronze/raw_csv"

checkpoint_path = "/Volumes/deltalakehouse/bronze/checkpoint"

schema_path = "/Volumes/deltalakehouse/bronze/schema"


# ============================================================
# 1. AUTO LOADER
# ============================================================

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)
)


# ============================================================
# 2. TIMESTAMP DA INGESTÃO
# ============================================================

df = df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)


# ============================================================
# 3. GRAVAR NA BRONZE
# ============================================================

query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True) # Obs.: para processamento continuo usar .trigger(processingTime="10 seconds")
    .toTable("deltalakehouse.bronze.dados")
)